In [2]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
import pandas as pd
from limb_fitting import *
from utils import *
from fitting import *
from interpolation import *
from reprojection import *

In [3]:
def show_data(image, to_file=None, **kwargs):
    if to_file is not None:
        plt.ioff()

    fig = plt.figure(figsize=(10,10))
    plt.imshow(image, **kwargs)
    plt.tight_layout()

    if to_file is not None:
        plt.savefig(to_file)
        plt.ion()
        plt.close(fig)


def get_scale(img_data):
    if img_data is not None:
        fmt, rng = img_data['PHI_IMG_format'], img_data['PHI_IMG_maxRange']

        scale = rng[-1] / rng[0]
        if fmt[-1] != 'IMGFMT_24_8':
            scale *= 256
        return scale
    else:
        return None


def CLD(mu, approximation='Neckel'):
    if approximation == 'Neckel':
        p = [0.48767921486914473,
             -1.6848471461910317,
             2.355950448408068,
             -1.827014432405401,
             1.3540877312885482,
             0.31414418403067174]
    else:
        a, b = -0.1, -0.45
        p = [a, -2 * a - b, a + b + 1]
    return np.polyval(p, mu) * (mu > 0)

In [4]:
df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/disk_centers_cor.csv', skipinitialspace=True).drop(columns='date').dropna()
dids = df['did'].to_numpy()
xu_sun, yu_sun, ru_sun = df['xu_sun'].to_numpy(), df['yu_sun'].to_numpy(), df['ru_sun'].to_numpy()

s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion_cor.npz')
xu, yu = s['xu'], s['yu']
xd, yd = s['xd'], s['yd']

In [5]:
dark_file = '/home/ulyanov/data/solo/phi/dark/solo_CAL1_phi-fdt-dark_20260126T223810_V202602200142C_0621261001.fits.gz'

with fits.open(dark_file) as hdul:
    dark_header = hdul[0].header
    dark = hdul[0].data

In [6]:
flat_file = '/home/ulyanov/data/solo/phi/flat/fdt/transmittance/phi-fdt-flat_20250310T080009_V202511271432C_0563100100.fits'

with fits.open(flat_file) as hdul:
    flat_header = hdul[0].header
    flat = hdul[0].data

In [15]:
files = sorted(glob.glob('/home/ulyanov/data/solo/phi/flare trigger/2026-02-04/*'))

In [16]:
x0, y0 = 560, 800
h0 = 128

Q = []

for file in files[:]:
    print(file)

    shifts = []

    with fits.open(file) as hdul:
        header = hdul[0].header
        data = hdul[0].data
        img_data = hdul['PHI_FITS_imageSummary'].data.copy()
        scale = get_scale(img_data)

    data = (data - scale * crop(dark, header)) / crop(flat[-1], header)

    for i in range(len(data)):
        temp = data[i]

        xc, yc, r_sun = find_center(temp)
        shifts += [(xc, yc)]

    shifts = np.array(shifts)

    Q_ = []
    for i in range(0,24,6):
        F = kll(np.log(data[i:i+6,x0-h0:x0+h0,y0-h0:y0+h0].clip(1)), shifts[i:i+6], niter=200, kind='quadratic')
        Q_ += [F]

    Q += [np.median(Q_, axis=0)]

Q = np.array(Q)

/home/ulyanov/data/solo/phi/flare trigger/2026-02-04/solo_L1_phi-fdt-ilam_20260204T120531_V202603281431C_0642040601.fits.gz
/home/ulyanov/data/solo/phi/flare trigger/2026-02-04/solo_L1_phi-fdt-ilam_20260204T120555_V202603281531C_0642040602.fits.gz
/home/ulyanov/data/solo/phi/flare trigger/2026-02-04/solo_L1_phi-fdt-ilam_20260204T120620_V202603281531C_0642040603.fits.gz
/home/ulyanov/data/solo/phi/flare trigger/2026-02-04/solo_L1_phi-fdt-ilam_20260204T120645_V202603281630C_0642040604.fits.gz
/home/ulyanov/data/solo/phi/flare trigger/2026-02-04/solo_L1_phi-fdt-ilam_20260204T120710_V202603281630C_0642040605.fits.gz
/home/ulyanov/data/solo/phi/flare trigger/2026-02-04/solo_L1_phi-fdt-ilam_20260204T120734_V202603290930C_0642040606.fits.gz
/home/ulyanov/data/solo/phi/flare trigger/2026-02-04/solo_L1_phi-fdt-ilam_20260204T120759_V202603291031C_0642040607.fits.gz
/home/ulyanov/data/solo/phi/flare trigger/2026-02-04/solo_L1_phi-fdt-ilam_20260204T120824_V202603291031C_0642040608.fits.gz
/home/ul

In [17]:
np.savez('FF2026-02-04.npz', logFF=Q, x0=x0, y0=y0)

In [18]:
plt.figure(figsize=(10,10))
plt.imshow(np.median(Q, axis=0), vmin=-5e-3, vmax=5e-3)

In [19]:
plt.figure(figsize=(10,10))
plt.imshow(np.median(np.append(Q, Q_, axis=0), axis=0), vmin=-5e-3, vmax=5e-3)

In [14]:
F = np.exp(np.median(np.append(Q, Q_, axis=0), axis=0))

In [13]:
plt.figure(figsize=(10,10))
plt.imshow(data[0])#, vmin=1.5e4)

In [15]:
plt.figure(figsize=(10,10))
plt.imshow(data[0,490:640,725:875] / F, vmin=1.5e4)

In [9]:
Q_ = Q.copy()

In [10]:
Q.shape

(10, 256, 256)